# Referencias ML Basicas v01

Este notebook deja preparadas referencias tabulares simples para contextualizar el LSTM. No reemplazan al modelo secuencial, pero ayudan a no evaluar la LSTM en el vacio.


## Referencias sugeridas

- `DummyClassifier`: linea base ingenua
- `LogisticRegression`: referencia lineal con balance de clases

Estas referencias son utiles para responder una pregunta academica importante: si el modelo secuencial agrega valor real o solo complejidad.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != 'frost_ema_igp' and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

PROJECT_ROOT


In [ ]:
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler

from src.data.make_dataset import build_and_save_datasets
from src.features.preprocessing import build_target_split_masks, impute_features_past_only, select_feature_columns

build_and_save_datasets()
processed_df = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'frost_dataset_v01.csv', index_col=0, parse_dates=True)
processed_df['target_timestamp'] = pd.to_datetime(processed_df['target_timestamp'])
feature_columns = select_feature_columns(processed_df.columns)
split_masks = build_target_split_masks(processed_df)

X, _ = impute_features_past_only(processed_df[feature_columns], split_masks['train'])
y = processed_df['frost_event_t_plus_12h']

scaler = StandardScaler().fit(X.loc[split_masks['train']])
X_train = scaler.transform(X.loc[split_masks['train']])
X_test = scaler.transform(X.loc[split_masks['test']])
y_train = y.loc[split_masks['train']]
y_test = y.loc[split_masks['test']]


In [ ]:
models = {
    'dummy_prior': DummyClassifier(strategy='prior'),
    'logistic_balanced': LogisticRegression(max_iter=500, class_weight='balanced')
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f'
===== {name} =====')
    print(classification_report(y_test, y_pred, digits=4))


## Alcance

No se optimizan hiperparametros aqui. El proposito es solo tener una referencia honesta y facil de explicar en la tesis o informe.
